# GPT-2 Text Generation with Curious Facts

In this notebook, we build a full GPT-2 pipeline to complete **curious fact prompts**.  
The idea is simple: provide the beginning of a fact-like question and let the model generate a plausible completion.

## Objective
1. Generate text with pretrained GPT-2.
2. Fine-tune GPT-2 on an English trivia dataset.
3. Compare generation quality before vs. after fine-tuning.
4. Add exploratory analysis and visual diagnostics to make the study more complete.

## Dataset
- `trivia_qa` (Hugging Face datasets)
- We use question-answer pairs and convert them into prompt-completion samples.

> Colab note: GPU is recommended for training.


## 0. Environment Setup

In [ ]:
# Run once in Colab
!pip -q install -U transformers datasets accelerate sentencepiece

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import inspect
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)


## 1. Load Pretrained GPT-2

In [ ]:
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print('Model:', model_name)
print('Vocab size:', tokenizer.vocab_size)
print('Parameters:', sum(p.numel() for p in model.parameters()))


In [ ]:
prompt = 'Curious fact question: Why do leaves change color in autumn?\nCurious fact completion:'

with torch.no_grad():
    tokens = tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)
    logits = model(input_ids=tokens).logits
    probs = torch.softmax(logits[0, -1, :], dim=-1)
    top_ids = torch.argsort(probs, descending=True)[:10]

print('Prompt:', prompt)
print('Input shape:', tokens.shape)
print('Output shape:', logits.shape)
print('\nTop next-token candidates:')
for tid in top_ids:
    print(f"{repr(tokenizer.decode(tid)):<16} -> {float(probs[tid])*100:.2f}%")


**Note:**
GPT predicts one token at a time. The final generation quality is strongly influenced by decoding strategy (greedy, temperature sampling, top-k/top-p, etc.).

## 2. Custom Generation Function (Epsilon-Greedy)

In [ ]:
def generate_custom(
    model: nn.Module,
    tokenizer,
    start: str,
    max_length: int = 60,
    eps: float = 0.5,
    top_n: int = 5,
    return_iterations: bool = False,
    device: str = 'cpu',
):
    parts = [start]
    rows = []

    with torch.no_grad():
        for _ in range(max_length):
            input_ids = tokenizer(''.join(parts), return_tensors='pt')['input_ids'].to(device)
            logits = model(input_ids=input_ids).logits
            probs = torch.softmax(logits[0, -1, :], dim=-1)
            sorted_tokens = torch.argsort(probs, descending=True)

            use_greedy = np.random.binomial(1, eps)
            if use_greedy:
                next_token = sorted_tokens[0]
                method = 'greedy'
            else:
                next_token = torch.multinomial(probs, 1)
                method = 'sample'

            token_text = tokenizer.decode(next_token)
            parts.append(token_text)

            if return_iterations:
                top_tokens = sorted_tokens[:top_n]
                rows.append({
                    'current_text': ''.join(parts),
                    'selected_token': token_text,
                    'method': method,
                    'top_tokens': [tokenizer.decode(t) for t in top_tokens],
                    'top_probs': [float(probs[t].cpu()) for t in top_tokens],
                })

    output_text = ''.join(parts)
    trace_df = pd.DataFrame(rows) if return_iterations else None
    return output_text, trace_df


In [ ]:
test_prompt = 'Curious fact question: Why is the sky blue?\nCurious fact completion:'

for eps in [1.0, 0.5, 0.2]:
    out, _ = generate_custom(model, tokenizer, test_prompt, max_length=50, eps=eps, device=device)
    print('\n' + '=' * 110)
    print(f'eps={eps}')
    print(out)


In [ ]:
# Hugging Face generation utility
inputs = tokenizer(test_prompt, return_tensors='pt').to(device)
max_len = inputs['input_ids'].shape[1] + 50

out = model.generate(
    **inputs,
    max_length=max_len,
    do_sample=True,
    temperature=0.8,
    top_p=0.92,
    pad_token_id=tokenizer.eos_token_id,
)

print(tokenizer.decode(out[0], skip_special_tokens=True))


**Note:**
- Higher exploitation (`eps` near 1.0) is usually more stable but repetitive.
- Higher exploration can be more creative but less precise.
- For fact completion, too much randomness often hurts factual consistency.

## 3. Load and Prepare Trivia Dataset

In [ ]:
# TriviaQA has multiple configs; we try robustly.
trivia_configs = ['rc.nocontext', 'unfiltered.nocontext']
raw = None
for cfg in trivia_configs:
    try:
        raw = load_dataset('trivia_qa', cfg)
        print('Loaded config:', cfg)
        break
    except Exception as e:
        print(f'Config {cfg} failed: {e}')

if raw is None:
    raise RuntimeError('Could not load trivia_qa with tested configs.')

raw


In [ ]:
def to_fact_sample(example):
    ans = example.get('answer', {})
    if isinstance(ans, dict):
        answer_text = str(ans.get('value', '')).strip()
        aliases = ans.get('aliases', [])
        if answer_text and answer_text not in aliases:
            aliases = [answer_text] + aliases
    else:
        answer_text = str(ans).strip()
        aliases = [answer_text] if answer_text else []

    question = str(example.get('question', '')).strip()
    prompt = f'Curious fact question: {question}\nCurious fact completion:'
    full_text = f'{prompt} {answer_text}'

    return {
        'question': question,
        'answer_text': answer_text,
        'aliases': aliases,
        'prompt': prompt,
        'full_text': full_text,
    }

processed = raw['train'].map(to_fact_sample, remove_columns=raw['train'].column_names)
processed = processed.filter(lambda x: len(x['question']) > 0 and len(x['answer_text']) > 0)

MAX_SAMPLES = 12000
if len(processed) > MAX_SAMPLES:
    processed = processed.shuffle(seed=SEED).select(range(MAX_SAMPLES))

split_ds = processed.train_test_split(test_size=0.1, seed=SEED)
split_ds


In [ ]:
split_ds['train'][0]


**Note:**
Each sample is reformatted into a causal-LM pattern:
- **Prompt:** question stem
- **Target completion:** answer text

This aligns directly with next-token prediction used by GPT models.

## 4. Exploratory Data Analysis

In [ ]:
df = split_ds['train'].to_pandas()
df['q_words'] = df['question'].str.split().apply(len)
df['a_words'] = df['answer_text'].str.split().apply(len)
df['q_first'] = df['question'].str.strip().str.split().str[0].str.lower().fillna('')

def answer_type(ans):
    a = str(ans).strip().lower()
    if re.search(r'\d', a):
        return 'contains_number'
    if any(m in a for m in ['january', 'february', 'march', 'april', 'may', 'june', 'july', 'august', 'september', 'october', 'november', 'december']):
        return 'date_like'
    if len(a.split()) <= 2:
        return 'short_entity'
    return 'long_phrase'

df['answer_type'] = df['answer_text'].apply(answer_type)

df[['q_words', 'a_words']].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])


In [ ]:
wh_order = ['who', 'what', 'where', 'when', 'which', 'how', 'why']
wh_counts = df['q_first'].value_counts()
wh_counts = pd.Series({k: int(wh_counts.get(k, 0)) for k in wh_order})

answer_type_counts = df['answer_type'].value_counts()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].hist(df['q_words'], bins=40, color='#457b9d', alpha=0.9)
axes[0, 0].set_title('Question Length Distribution')
axes[0, 0].set_xlabel('Words per question')
axes[0, 0].set_ylabel('Frequency')

axes[0, 1].hist(df['a_words'], bins=35, color='#2a9d8f', alpha=0.9)
axes[0, 1].set_title('Answer Length Distribution')
axes[0, 1].set_xlabel('Words per answer')
axes[0, 1].set_ylabel('Frequency')

axes[1, 0].bar(wh_counts.index, wh_counts.values, color='#264653')
axes[1, 0].set_title('Question Start Word (WH)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].tick_params(axis='x', rotation=35)

axes[1, 1].bar(answer_type_counts.index, answer_type_counts.values, color='#e76f51')
axes[1, 1].set_title('Heuristic Answer Type Distribution')
axes[1, 1].set_ylabel('Count')
axes[1, 1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()


In [ ]:
# Top frequent content words in questions (quick lexical view)
def tokenize_alpha(text):
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if len(w) > 2]

stop = {
    'the', 'and', 'for', 'that', 'with', 'from', 'this', 'what', 'when', 'where', 'which',
    'who', 'how', 'why', 'was', 'were', 'are', 'have', 'has', 'had', 'into', 'about', 'after',
    'before', 'under', 'over', 'between', 'than', 'then', 'their', 'there', 'they', 'them'
}

counter = Counter()
for q in df['question'].sample(min(5000, len(df)), random_state=SEED):
    words = [w for w in tokenize_alpha(q) if w not in stop]
    counter.update(words)

top_words = pd.DataFrame(counter.most_common(20), columns=['word', 'count'])

top_words.head(20)


In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(top_words['word'], top_words['count'], color='#1d3557')
plt.title('Top 20 Content Words in Questions')
plt.xticks(rotation=50, ha='right')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


**Note:**
The dataset mostly contains short factual answers and WH-style questions. This supports the completion objective, since the model can learn concise fact continuations rather than long narrative text.

## 5. Tokenization and Training Data

In [ ]:
def preprocess_function(examples, max_len=128):
    return tokenizer(examples['full_text'], truncation=True, max_length=max_len)

tokenized_ds = split_ds.map(preprocess_function, batched=True)
keep_cols = {'input_ids', 'attention_mask'}
remove_cols = [c for c in tokenized_ds['train'].column_names if c not in keep_cols]
tokenized_ds = tokenized_ds.remove_columns(remove_cols)
tokenized_ds.set_format('torch')

tokenized_ds


**Note:**
For causal language modeling, we only need token sequences (`input_ids`, optional `attention_mask`). The target is the same sequence shifted by one token internally.

## 6. Baseline Completion Before Fine-Tuning

In [ ]:
eval_subset = split_ds['test'].shuffle(seed=SEED).select(range(min(40, len(split_ds['test']))))


def generate_completion(model, tokenizer, prompt, max_new_tokens=24, temperature=0.7, top_p=0.9):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    max_len = inputs['input_ids'].shape[1] + max_new_tokens

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_length=max_len,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    if decoded.startswith(prompt):
        return decoded[len(prompt):].strip()
    return decoded.strip()

baseline_rows = []
for ex in eval_subset:
    gen = generate_completion(model, tokenizer, ex['prompt'])
    baseline_rows.append({
        'question': ex['question'],
        'prompt': ex['prompt'],
        'expected': ex['answer_text'],
        'aliases': ex['aliases'],
        'generated_before': gen,
    })

baseline_df = pd.DataFrame(baseline_rows)
baseline_df[['question', 'expected', 'generated_before']].head(10)


In [ ]:
def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9 ]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def alias_hit(generated, aliases):
    g = normalize_text(generated)
    valid_aliases = [a for a in aliases if isinstance(a, str) and a.strip()]
    valid_aliases = valid_aliases[:15]  # keep runtime controlled
    return int(any(normalize_text(a) in g for a in valid_aliases))

baseline_df['hit_before'] = baseline_df.apply(lambda r: alias_hit(r['generated_before'], r['aliases']), axis=1)
baseline_hit_rate = baseline_df['hit_before'].mean()
print('Baseline hit rate:', round(float(baseline_hit_rate), 4))


**Note:**
The hit-rate metric is intentionally simple: a generated completion is considered successful if it contains any known answer alias. It is not perfect, but it gives a quick quantitative baseline.

## 7. Fine-Tuning GPT-2

In [ ]:
import transformers
print('Transformers version:', transformers.__version__)

batch_size = 8 if torch.cuda.is_available() else 2
logging_steps = max(1, len(tokenized_ds['train']) // batch_size // 5)

sig = inspect.signature(TrainingArguments.__init__)
valid_args = set(sig.parameters.keys())

cfg = {
    'output_dir': './gpt2-trivia-curious-facts',
    'num_train_epochs': 2,
    'learning_rate': 2e-5,
    'per_device_train_batch_size': batch_size,
    'per_device_eval_batch_size': batch_size,
    'weight_decay': 0.01,
    'logging_steps': logging_steps,
}

if 'overwrite_output_dir' in valid_args:
    cfg['overwrite_output_dir'] = True

if 'evaluation_strategy' in valid_args:
    cfg['evaluation_strategy'] = 'epoch'
elif 'eval_strategy' in valid_args:
    cfg['eval_strategy'] = 'epoch'

if 'save_strategy' in valid_args:
    cfg['save_strategy'] = 'epoch'

if 'load_best_model_at_end' in valid_args:
    cfg['load_best_model_at_end'] = True

if 'report_to' in valid_args:
    cfg['report_to'] = 'none'

if 'disable_tqdm' in valid_args:
    cfg['disable_tqdm'] = True

if 'fp16' in valid_args:
    cfg['fp16'] = torch.cuda.is_available()

training_args = TrainingArguments(**cfg)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['test'],
)

trainer


In [ ]:
%%time
train_output = trainer.train()
train_output


In [ ]:
try:
    eval_metrics = trainer.evaluate()
except RuntimeError as e:
    if 'on_train_begin must be called before on_evaluate' in str(e):
        # Notebook progress callback can fail in some Colab/Jupyter execution orders.
        try:
            from transformers.utils.notebook import NotebookProgressCallback
            trainer.remove_callback(NotebookProgressCallback)
        except Exception:
            pass
        eval_metrics = trainer.evaluate()
    else:
        raise

eval_metrics

In [ ]:
if 'eval_loss' in eval_metrics:
    ppl = float(np.exp(eval_metrics['eval_loss']))
    print('Approx perplexity:', round(ppl, 2))
else:
    print('eval_loss not available in metrics')


In [ ]:
log_df = pd.DataFrame(trainer.state.log_history)

plt.figure(figsize=(8, 4))
if 'loss' in log_df.columns:
    train_curve = log_df[log_df['loss'].notna()]
    plt.plot(train_curve['step'], train_curve['loss'], label='train loss')
if 'eval_loss' in log_df.columns:
    eval_curve = log_df[log_df['eval_loss'].notna()]
    plt.plot(eval_curve['step'], eval_curve['eval_loss'], marker='o', label='eval loss')

plt.title('Training Curves')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(alpha=0.25)
plt.legend()
plt.show()


**Note:**
If training loss improves but eval loss plateaus or worsens, this suggests overfitting. In that case, reduce epochs, increase data, or tune decoding during inference.

## 8. Completion Quality After Fine-Tuning

In [ ]:
baseline_df['generated_after'] = baseline_df['prompt'].apply(lambda p: generate_completion(model, tokenizer, p))
baseline_df['hit_after'] = baseline_df.apply(lambda r: alias_hit(r['generated_after'], r['aliases']), axis=1)

after_hit_rate = baseline_df['hit_after'].mean()
print('Hit rate before fine-tuning:', round(float(baseline_hit_rate), 4))
print('Hit rate after fine-tuning :', round(float(after_hit_rate), 4))

baseline_df[['question', 'expected', 'generated_before', 'generated_after', 'hit_before', 'hit_after']].head(12)


In [ ]:
rates = pd.DataFrame({
    'stage': ['before', 'after'],
    'hit_rate': [float(baseline_hit_rate), float(after_hit_rate)]
})

plt.figure(figsize=(5, 4))
plt.bar(rates['stage'], rates['hit_rate'], color=['#8ecae6', '#219ebc'])
plt.ylim(0, 1)
plt.title('Alias Hit Rate Comparison')
plt.ylabel('Hit rate')
for i, v in enumerate(rates['hit_rate']):
    plt.text(i, v + 0.02, f'{v:.2f}', ha='center')
plt.tight_layout()
plt.show()


In [ ]:
# Show a few qualitative examples
sample_view = baseline_df.sample(min(6, len(baseline_df)), random_state=SEED)
for _, row in sample_view.iterrows():
    print('\n' + '=' * 120)
    print('QUESTION:', row['question'])
    print('EXPECTED:', row['expected'])
    print('BEFORE :', row['generated_before'])
    print('AFTER  :', row['generated_after'])
    print('HIT BEFORE/AFTER:', row['hit_before'], '/', row['hit_after'])


**Note:**
For this task, qualitative inspection is still important. A completion can miss alias matching but remain semantically close, and the opposite can also happen.

## 9. Conclusions

1. GPT-2 can generate coherent continuations for trivia-style prompts, but factual correctness is still inconsistent.
2. The model learned the prompt format (`Curious fact question` -> `Curious fact completion`) and produced cleaner completion structure.
3. Quantitative results (alias hit rate) were modest in this run, showing that better text fluency does not always imply higher exact-answer matching.
4. Qualitative inspection remains necessary: some completions are plausible but miss literal aliases used by the metric.
5. Evaluation is sensitive to decoding settings and sample size, so conclusions should be interpreted with those limits in mind.